In [1]:
"""
IEOR 4732 – Computational Methods in Finance
Case Study 2: Up-and-Out Call (UOC) via PIDE
Pranavi Parsi
UNI pp2955
3/16/26
"""

import numpy as np
from scipy.special import gammaincc, gamma as sp_gamma
import warnings
from scipy.integrate import quad
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd, json
warnings.filterwarnings('ignore')

In [2]:
def levy_params(sigma, nu, theta):
    """
    Given lambda tails 
    """
    disc = np.sqrt(theta**2/sigma**4 + 2/(sigma**2*nu))
    lp = disc - theta/sigma**2
    ln = disc + theta/sigma**2
    return lp, ln

In [3]:
def g1(xi, alpha):
    """
    From lecture slides:
    g1(xi, alpha) = Integral from xi to infinity of (e^-z/z^alpha) dz
    Valid for alpha in (0,1).
    """
    a = 1.0 - alpha
    safe = np.where(xi < 1e-14, 1e-14, xi)
    return gammaincc(a, safe) * sp_gamma(a)


def g2(xi, alpha):
    """
    From lecture slides:
    g2(xi, alpha) = Integral from xi to infinity of (e^-z/z^{alpha+1}) dz
    """
    safe = np.where(xi < 1e-14, 1e-14, xi)
    return np.exp(-safe) / (alpha * safe**alpha) - g1(safe, alpha) / alpha

In [4]:
def sigma2_e(eps, lp, ln, nu, Y):
    """
    From lecture slides:
    sigma^2(e) = 
      sum over lp, ln on (lam^{Y-2}/nu) * [-(lam*e)^{1-Y}*exp(-lam*e) + (1-Y)*(g1(0) - g1(lam*e, Y))]
    """
    def contrib(lam):
        t1 = (lam**(Y-2) / nu)
        le = lam * eps
        t2 = -(le**(1.0-Y))*np.exp(-le)
        t3 = (1.0-Y)*(g1(0, Y)-g1(le, Y))
        
        return  t1 * (t2 + t3)
    return float(contrib(lp) + contrib(ln))
 
 
def omega_e(eps, lp, ln, nu, Y):
    """
    From lecture slides:
    omega(eps) = 
        sum over lp, ln on (lam^Y/nu)*g2(lam*eps,Y) - ((lam-1)^Y/nu)*g2((lam-1)*eps,Y)
    
    This corrects the drift so that the discounted stock price is a martingale
    under the risk-neutral measure:
        E^Q[S_t] = S_0 * exp((r-q)*t) iff mu = r-q+omega-sigma^2/2 in PIDE
    """
    t1 = (lp**Y/nu)*g2(lp*eps,Y)
    t2 = ((lp-1)**Y/nu)*g2((lp-1)*eps,Y) if lp > 1.0 else 0.0
    t3 = (ln**Y/nu)*g2(ln*eps,Y)
    t4 = ((ln+1)**Y/nu)*g2((ln+1)*eps,Y)
    return float(t1 - t2 + t3 - t4)

In [5]:
def precompute_g(N, dx, lp, ln, Y):
    """
    From lecture slides:
    Pre-calculate g1 and g2 vectors at integer multiples of dx for k=1..N+1
    """
    k = np.arange(0, N+2, dtype=float)
    G = {}
    for lam, tag in [(lp, 'p'), (ln, 'n')]:
        xi = np.where(k*dx*lam < 1e-14, 1e-14, k*dx*lam)
        G[f'g1_{tag}'] = g1(xi,Y)
        G[f'g2_{tag}'] = g2(xi,Y)
    xi_np1 = np.where(k*dx*(ln+1) < 1e-14, 1e-14, k*dx*(ln+1))
    G['g2_np1'] = g2(xi_np1,Y)
    if lp > 1.0:
        xi_pm1 = np.where(k*dx*(lp-1) < 1e-14, 1e-14, k*dx*(lp-1))
        G['g2_pm1'] = g2(xi_pm1,Y)
    else:
        G['g2_pm1'] = np.zeros(N+2)
    return G

In [6]:
def build_integral_matrix(N, dx, lp, ln, nu, Y, x_grid, G):
    """
    From lecture slides:
    For interior node i, the integral is split into four regions:

    (a) Positive inside grid: y in (dx, x_N-x_i), k=1..N-i-1
        w = (lp^Y/nu)*[(w_{i+k}-w_i - k*(w_{i+k+1}-w_{i+k}))*(g2(k*lp*dx)-g2((k+1)*lp*dx))
                    + (w_{i+k+1}-w_{i+k})/(lp*dx)*(g1(k*lp*dx)-g1((k+1)*lp*dx))]
    (b) Negative inside grid: y in (x_0-x_i, -dx), k=1..i-1
        Same formula with ln and reflected indices.
    (c) Negative outside grid: y < x_0-x_i
        w = -(ln^Y/nu)*w_i * g2(i*ln*dx, Y)
    (d) Positive outside grid: y > x_N-x_i
        w = -(lp^Y/nu)*w_i * g2((N-i)*lp*dx, Y)
    """
    n_int = N - 1
    J = np.zeros((n_int, N+1))
    bdy = np.zeros(n_int)

    for i in range(1, N):
        row = i - 1

        # Positive inside grid
        for k in range(1, N-i):
            a = k * lp * dx
            b = (k+1) * lp * dx
            dg2 = G['g2_p'][k]-G['g2_p'][k+1]
            dg1 = G['g1_p'][k]-G['g1_p'][k+1]
            cp = lp**Y / nu
            J[row, i+k] += cp*(1+k)*dg2
            J[row, i] += cp*(-1.0)*dg2
            if i+k+1 <= N:
                J[row, i+k+1] += cp*(-k)*dg2
                c2 = cp/(lp * dx)
                J[row, i+k+1] += c2*dg1
                J[row, i+k] -= c2*dg1

        # Negative inside grid
        for k in range(1, i):
            a = k*ln*dx
            b = (k+1)*ln*dx
            dg2 = G['g2_n'][k]-G['g2_n'][k+1]
            dg1 = G['g1_n'][k]-G['g1_n'][k+1]
            cn = ln**Y/nu
            J[row, i-k] += cn*(1+k)*dg2
            J[row, i] += cn*(-1.0)*dg2
            if i-k-1 >= 0:
                J[row, i-k-1] += cn*(-k)*dg2
                c2 = cn/(ln*dx)
                J[row, i-k-1] += c2*dg1
                J[row, i-k] -= c2*dg1

        # Negative outside grid
        J[row, i] -= (ln**Y/nu)*G['g2_n'][i]

        # Positive outside grid
        J[row, i] -= (lp**Y/nu)*G['g2_p'][N-i]

    return J, bdy

In [7]:
def thomas(diag, lower, upper, rhs):
    n = len(diag)
    c = upper.copy()
    d = diag.copy()
    b = rhs.copy()
    for i in range(1, n):
        m = lower[i-1]/d[i-1]
        d[i] -= m*c[i-1]
        b[i] -= m*b[i-1]
    x = np.zeros(n)
    x[-1] = b[-1]/d[-1]
    for i in range(n-2, -1, -1):
        x[i] = (b[i]-c[i]*x[i+1])/d[i]
    return x

In [8]:
def solve_uoc_pide(S0, K, B, r, q, T, sigma, nu, theta, Y,
                   N=120, M=120, scheme='explicit-implicit'):
    """
    Solves the UOC backward PIDE.
    """
    lp, ln = levy_params(sigma, nu, theta)

    x_max = np.log(B)
    x_min = np.log(S0) - 5.0*sigma*np.sqrt(T)
    dx = (x_max-x_min)/N
    dt = T/M
    x_grid = x_min + np.arange(N+1, dtype=float)*dx

    w = np.maximum(np.exp(x_grid) - K, 0.0)
    w[0] = 0.0 #given
    w[-1] = 0.0 #given

    sig2 = sigma2_e(dx, lp, ln, nu, Y)
    omg = omega_e(dx, lp, ln, nu, Y)
    drift = r-q+omg-0.5*sig2

    Bl = sig2*dt/(2*dx**2) - drift*dt/(2*dx)
    Bu = sig2*dt/(2*dx**2) + drift*dt/(2*dx)

    n_int = N - 1

    diag = np.full(n_int, 1.0+r*dt+Bl+Bu)
    lower = np.full(n_int-1, -Bl)
    upper = np.full(n_int-1, -Bu)

    # Half-step matrices for Crank-Nicolson
    Bl_h, Bu_h = 0.5*Bl, 0.5*Bu
    diag_imp = np.full(n_int, 1.0+ 0.5*r*dt + Bl_h + Bu_h)
    lower_imp = np.full(n_int-1, -Bl_h)
    upper_imp = np.full(n_int-1, -Bu_h)
    diag_exp = np.full(n_int, 1.0 - 0.5*r*dt - Bl_h - Bu_h)
    upper_exp = np.full(n_int-1, Bu_h)
    lower_exp = np.full(n_int-1, Bl_h)

    G = precompute_g(N, dx, lp, ln, Y)
    J, bdy = build_integral_matrix(N, dx, lp, ln, nu, Y, x_grid, G)

    for _ in range(M):
        w_old = w.copy()
        w_old[0] = 0.0
        w_old[-1] = 0.0
        int_rhs = J @ w_old + bdy
        
        if scheme == 'explicit-implicit':
            rhs = w_old[1:N] + dt * int_rhs
            rhs[0] += Bl * w_old[0]
            rhs[-1] += Bu * w_old[-1]
            w[1:N] = thomas(diag.copy(), lower.copy(), upper.copy(), rhs)

        else:  # Crank-Nicolson
            w_int = w_old[1:N]
            rhs_cn = diag_exp*w_int
            rhs_cn[:-1] += upper_exp*w_int[1:]
            rhs_cn[1:] += lower_exp*w_int[:-1]
            rhs_cn += dt*int_rhs
            rhs_cn[0] += Bl_h*w_old[0]
            rhs_cn[-1] += Bu_h*w_old[-1]
            w[1:N] = thomas(diag_imp.copy(), lower_imp.copy(), upper_imp.copy(), rhs_cn)

        w[0] = 0.0;  w[-1] = 0.0
        w = np.maximum(w, 0.0)

    x0  = np.log(float(S0))
    idx = np.clip(int(np.searchsorted(x_grid, x0))-1, 0, N-1)
    frac = (x0 - x_grid[idx])/dx
    price = w[idx] * (1.0-frac) + w[idx+1]*frac

    return float(price), w, x_grid

In [9]:
def cgmy_vanilla_call(S0, K, r, q, T, sigma, nu, theta, Y):
    """
    European call price via Lewis (2001) Fourier transform for CGMY process.
    Used to sanity check UOC <= vanilla call.
    """
    lp, ln = levy_params(sigma, nu, theta)
    C_cgmy = 1.0 / nu
    
    omega_rn = -C_cgmy*sp_gamma(-Y)*((ln-1)**Y-ln**Y+(lp+1)**Y-lp**Y)

    def cf(u):
        psi = C_cgmy*sp_gamma(-Y)*((ln - 1j*u)**Y-ln**Y+(lp+1j*u)**Y-lp**Y)
        return np.exp(1j * u * (r - q + omega_rn) * T + psi * T)

    k_log = np.log(K/S0)
    alpha = 1.5  # damping parameter

    def integrand(v):
        u = v-1j*alpha
        phi = cf(u)
        num = np.exp(-1j*v*k_log)*phi
        denom = alpha**2+alpha-v**2+1j*(2*alpha+1)*v
        return np.real(num / denom)

    I, _ = quad(integrand, 0, 200, limit=400)
    return float(S0*np.exp(-alpha*k_log-r*T)/np.pi*I)


In [10]:
def run_convergence(base_params, grid_sizes):
    results = []
    for G in grid_sizes:
        print(f"  N=M={G}")
        p_ei,_,_ = solve_uoc_pide(**base_params, N=G, M=G, scheme='explicit-implicit')
        p_cn,_,_ = solve_uoc_pide(**base_params, N=G, M=G, scheme='crank-nicolson')
        results.append({'N=M': G, 'Exp-Imp': p_ei, 'CN': p_cn})
    return results


def run_sensitivity(base_params, param, values, N=100, M=100):
    rows = []
    for v in values:
        p = {**base_params, param: v}
        p_ei,_,_ = solve_uoc_pide(**p, N=N, M=M, scheme='explicit-implicit')
        p_cn,_,_ = solve_uoc_pide(**p, N=N, M=M, scheme='crank-nicolson')
        rows.append({param: v, 'Exp-Imp': p_ei, 'CN': p_cn})
    return rows

In [11]:
#given
base = dict(S0=1900, K=2000, B=2200, r=0.0025, q=0.015, T=0.5, sigma=0.25, nu=0.31, theta=-0.25, Y=0.4)
lp_v, ln_v = levy_params(base['sigma'], base['nu'], base['theta'])
print(f"Lévy parameters: lambda_p={lp_v:.4f}, lambda_n={ln_v:.4f}")
print(f"C = 1/nu = {1/base['nu']:.4f}")

# Benchmark: CN at fine grid (N=M=200)
# Crank-Nicolson at the finest grid is our primary reference value because there is no process approximation.
print("\nBenchmark: Crank-Nicolson at N=M=200")
p_cn_ref, w_cn_ref, xg_ref = solve_uoc_pide(**base, N=200, M=200, scheme='crank-nicolson')
print(f"  CN benchmark (N=M=200): {p_cn_ref:.6f}")

# Fourier upper bound (vanilla CGMY call, no barrier)
vc = cgmy_vanilla_call(**{k: v for k, v in base.items() if k != 'B'})
print(f"  CGMY vanilla call (Fourier upper bound): {vc:.4f}")
print(f"  Benchmark / Vanilla ratio: {p_cn_ref/vc:.2%}")

# Explicit-Implicit finite difference scheme at matching grid, compare to benchmark
print("\nEI at N=M=200, error with CN benchmark")
p_ei_200, w_ei_200, _ = solve_uoc_pide(**base, N=200, M=200, scheme='explicit-implicit')
diff = p_ei_200 - p_cn_ref
print(f"  EI (N=M=200): {p_ei_200:.6f}")
print(f"  CN (N=M=200): {p_cn_ref:.6f} [BENCHMARK]")
print(f"  EI − CN     : {diff:+.6f}  ({100*diff/p_cn_ref:+.3f}%)")

# Grid convergencefor EI vs CN Benchmark
print("\nGrid convergence EI vs CN benchmark")
conv = run_convergence(base, [40, 60, 80, 100, 120, 150, 200])
df_c = pd.DataFrame(conv)
df_c['EI |err|'] = (df_c['Exp-Imp'] - p_cn_ref).abs()
df_c['CN |err|'] = (df_c['CN'] - p_cn_ref).abs()
df_c['EI err %'] = 100 * df_c['EI |err|'] / p_cn_ref
df_c['CN err %'] = 100 * df_c['CN |err|'] / p_cn_ref
print(df_c.to_string(index=False, float_format=lambda x: f"{x:.5f}"))

# Sensitivities: EI vs CN
def sens_with_error(rows):
    df = pd.DataFrame(rows)
    df['EI − CN'] = df['Exp-Imp'] - df['CN']
    df['EI err %'] = 100 * (df['Exp-Imp'] - df['CN']).abs() / df['CN']
    return df

print("\nSensitivity: sigma")
s_sig = run_sensitivity(base, 'sigma', [0.15, 0.20, 0.25, 0.30, 0.35])
print(sens_with_error(s_sig).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nSensitivity: Barrier B")
s_B = run_sensitivity(base, 'B', [2050, 2100, 2150, 2200, 2300])
print(sens_with_error(s_B).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nSensitivity: Maturity T")
s_T = run_sensitivity(base, 'T', [0.25, 0.50, 0.75, 1.00])
print(sens_with_error(s_T).to_string(index=False,
      float_format=lambda x: f"{x:.4f}"))

print("\nSensitivity: Jump activity Y")
s_Y = run_sensitivity(base, 'Y', [0.1, 0.2, 0.3, 0.4, 0.5, 0.6])
print(sens_with_error(s_Y).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nSensitivity: nu (variance rate)")
s_nu = run_sensitivity(base, 'nu', [0.10, 0.20, 0.31, 0.50, 0.80])
print(sens_with_error(s_nu).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nSensitivity: theta (skew)")
s_th = run_sensitivity(base, 'theta', [-0.40, -0.25, -0.10, 0.0, 0.10])
print(sens_with_error(s_th).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Plots
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("IEOR 4732 Case Study 2 – UOC PIDE Results", fontsize=13, fontweight='bold')

S_v  = np.exp(xg_ref)
mask = (S_v >= 1400) & (S_v <= 2200)
ax = axes[0, 0]
ax.plot(S_v[mask], w_ei_200[mask], 'b-', lw=2, label=f'EI  ({p_ei_200:.2f})')
ax.plot(S_v[mask], w_cn_ref[mask], 'r--', lw=2, label=f'CN  ({p_cn_ref:.2f}) ← benchmark')
ax.axvline(base['K'], color='grey', ls=':', lw=1.2, label=f"K={base['K']}")
ax.axvline(base['B'], color='orange', ls=':', lw=1.2, label=f"B={base['B']}")
ax.scatter([base['S0']], [p_cn_ref], s=80, color='red', zorder=5)
ax.scatter([base['S0']], [p_ei_200], s=80, color='blue', zorder=5)
ax.set(xlabel='Stock Price S', ylabel='Option Value', title='UOC Value Profile at τ=T  (N=M=200)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.plot(df_c['N=M'], df_c['Exp-Imp'], 'bo-', label='EI')
ax.plot(df_c['N=M'], df_c['CN'], 'rs-', label='CN')
ax.axhline(p_cn_ref, color='darkred', ls='--', lw=1.8, label=f'CN benchmark={p_cn_ref:.4f}')
ax.set(xlabel='Grid size N=M', ylabel='UOC Price', title='Grid Convergence')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[0, 2]
df_plot = df_c[df_c['N=M'] < 200]
ax.semilogy(df_plot['N=M'], df_plot['EI |err|'], 'bo-', label='EI vs CN bmark')
ax.semilogy(df_plot['N=M'], df_plot['CN |err|'], 'rs-', label='CN vs CN bmark')
ax.set(xlabel='N=M', ylabel='|price − CN benchmark|', title='Abs Error vs CN Benchmark')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 0]
df_s = pd.DataFrame(s_sig)
ax.plot(df_s['sigma'], df_s['Exp-Imp'], 'bo-', label='EI')
ax.plot(df_s['sigma'], df_s['CN'], 'rs-', label='CN (benchmark)')
ax.fill_between(df_s['sigma'], df_s['Exp-Imp'], df_s['CN'], alpha=0.15, color='purple', label='EI−CN gap')
ax.set(xlabel='σ', ylabel='UOC Price', title='Sensitivity: sigma')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 1]
df_b = pd.DataFrame(s_B)
ax.plot(df_b['B'], df_b['Exp-Imp'], 'bo-', label='EI')
ax.plot(df_b['B'], df_b['CN'], 'rs-', label='CN (benchmark)')
ax.fill_between(df_b['B'], df_b['Exp-Imp'], df_b['CN'], alpha=0.15, color='purple', label='EI−CN gap')
ax.set(xlabel='Barrier B', ylabel='UOC Price', title='Sensitivity: Barrier B')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 2]
df_y = pd.DataFrame(s_Y)
ax.plot(df_y['Y'], df_y['Exp-Imp'], 'bo-', label='EI')
ax.plot(df_y['Y'], df_y['CN'], 'rs-', label='CN (benchmark)')
ax.fill_between(df_y['Y'], df_y['Exp-Imp'], df_y['CN'], alpha=0.15, color='purple', label='EI−CN gap')
ax.set(xlabel='Y (jump activity)', ylabel='UOC Price', title='Sensitivity: Jump Activity Y')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/Users/pparsi27/Downloads/uoc_pide_results.png', dpi=150, bbox_inches='tight')
print("\nPlots saved")

summary = {
    'base_ei_200': p_ei_200,
    'base_cn_200': p_cn_ref,
    'vanilla_fft': vc,
    'lp': lp_v, 'ln': ln_v,
    'conv': conv,
    's_sig': s_sig, 's_B': s_B, 's_T': s_T,
    's_Y': s_Y, 's_nu': s_nu, 's_th': s_th,
}
with open('/Users/pparsi27/Downloads/results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("Summary saved")

Lévy parameters: lambda_p=14.9191, lambda_n=6.9191
C = 1/nu = 3.2258

Benchmark: Crank-Nicolson at N=M=200
  CN benchmark (N=M=200): 4.265150
  CGMY vanilla call (Fourier upper bound): 115.2218
  Benchmark / Vanilla ratio: 3.70%

EI at N=M=200, error with CN benchmark
  EI (N=M=200): 4.221839
  CN (N=M=200): 4.265150 [BENCHMARK]
  EI − CN     : -0.043311  (-1.015%)

Grid convergence EI vs CN benchmark
  N=M=40
  N=M=60
  N=M=80
  N=M=100
  N=M=120
  N=M=150
  N=M=200
 N=M  Exp-Imp      CN  EI |err|  CN |err|  EI err %  CN err %
  40  3.21897 3.31575   1.04618   0.94940  24.52866  22.25954
  60  3.59045 3.68235   0.67470   0.58280  15.81897  13.66427
  80  3.79533 3.87665   0.46982   0.38850  11.01532   9.10880
 100  3.92567 3.99732   0.33948   0.26783   7.95938   6.27938
 120  4.01580 4.07943   0.24935   0.18572   5.84630   4.35436
 150  4.11609 4.17032   0.14906   0.09483   3.49484   2.22336
 200  4.22184 4.26515   0.04331   0.00000   1.01546   0.00000

Sensitivity: sigma
 sigma  Exp-